In [53]:
import random
from dotenv import load_dotenv
import os
import re
from openai import OpenAI

In [54]:
# Model setup (OpenAI, GitHub-hosted GPT-4.1)
load_dotenv()
token = os.getenv("OpenAIKey")
if not token:
    raise ValueError("GITHUB_AI_TOKEN not found in environment variables.")
endpoint = "https://models.github.ai/inference"
model = "openai/gpt-4.1"

In [55]:
client = OpenAI(
    base_url=endpoint,
    api_key=token,
)

In [56]:
# Sim parameters
PRODUCTS = {
    "Red Bull": {"cost": 1.50, "price": 2.50, "inventory": 1},
    "Snickers": {"cost": 0.50, "price": 1.25, "inventory": 1}
}
DAILY_FEE = 5.0
START_BALANCE = 100.0
DAYS = 3

In [57]:
# helper functions
def simulate_sales(product, inventory, day):
    # Simulate customer purchases (weekend effect)
    base_sales = random.randint(2, 7)
    if day % 7 in [5, 6]:  # Weekend
        base_sales += random.randint(2, 5)
    return min(base_sales, inventory)

def agent_decision(state):
    user_msg = (
        f"You are running a vending machine. Here is today's state:\n"
        f"Balance: ${state['balance']:.2f}\n"
        f"Inventory: {state['inventory']}\n"
        f"Prices: {state['prices']}\n"
        f"Daily Fee: ${DAILY_FEE}\n"
        f"Yesterday's sales: {state['sales']}\n"
        "For today's actions, you must specify exact commands using this format:\n"
        "- To restock, write: restock [product] [amount] (e.g. 'restock Red Bull 10').\n"
        "- To change price, write: set [product] price to [value] (e.g. 'set Snickers price to 1.00').\n"
        "You can include multiple actions in one sentence, separated by commas. Only use these formats, and keep your response to one sentence."
    )
    response = client.chat.completions.create(
        messages=[
            {"role": "system", "content": ""},
            {"role": "user", "content": user_msg}
        ],
        temperature=1,
        top_p=1,
        model=model
    )
    return response.choices[0].message.content

def run_simulation():
    state = {
        "balance": START_BALANCE,
        "inventory": {k: v["inventory"] for k, v in PRODUCTS.items()},
        "prices": {k: v["price"] for k, v in PRODUCTS.items()},
        "sales": {k: 0 for k in PRODUCTS.keys()}
    }
    TRACE = []

    for day in range(1, DAYS + 1):
        day_sales = {}
        for product in PRODUCTS:
            sold = simulate_sales(product, state["inventory"][product], day)
            state["inventory"][product] -= sold
            state["balance"] += sold * state["prices"][product]
            day_sales[product] = sold

        # Pay daily fee
        state["balance"] -= DAILY_FEE

        # Log trace
        trace_entry = {
            "day": day,
            "balance": state["balance"],
            "inventory": state["inventory"].copy(),
            "sales": day_sales.copy(),
            "prices": state["prices"].copy()
        }

        # Agent decision
        agent_action = agent_decision({
            "balance": state["balance"],
            "inventory": state["inventory"],
            "sales": day_sales,
            "prices": state["prices"]
        })
        trace_entry["agent_action"] = agent_action

        # Parse agent_action for restocks and price changes
        restock_dict = {}
        price_updates = {}
        emails_sent = []

        for product in PRODUCTS:
            product_lower = product.lower()

            # Restock parsing: e.g. "restock Red Bull 12"
            restock_match = re.search(rf"restock {product_lower} (\d+)", agent_action.lower())
            if restock_match:
                qty = int(restock_match.group(1))
                cost = PRODUCTS[product]["cost"] * qty
                if state["balance"] >= cost:
                    state["inventory"][product] += qty
                    state["balance"] -= cost
                    restock_dict[product] = qty
                else:
                    restock_dict[product] = 0
                if "email" in agent_action.lower() or "contact" in agent_action.lower():
                    emails_sent.append(
                        f"Agent requests approval to restock {product}: {qty} units at ${PRODUCTS[product]['cost']} each."
                    )
            else:
                restock_dict[product] = 0

            # Price change parsing: e.g. "set Red Bull price to 3.00"
            price_match = re.search(rf"set {product_lower} price to ([\d.]+)", agent_action.lower())
            if price_match:
                new_price = float(price_match.group(1))
                state["prices"][product] = new_price
                price_updates[product] = new_price

        trace_entry["restock"] = restock_dict
        if price_updates:
            trace_entry["new_prices"] = price_updates
        if emails_sent:
            trace_entry["emails_sent"] = emails_sent

        TRACE.append(trace_entry)

        if state["balance"] < 0:
            print(f"Bankrupt on day {day}")
            break

    return TRACE

In [58]:
if __name__ == "__main__":
    trace = run_simulation()
    for day in trace:
        print(f"Day {day['day']}:")
        print(f"  Balance: ${day['balance']:.2f}")
        print(f"  Inventory: {day['inventory']}")
        print(f"  Prices: {day['prices']}")
        print(f"  Sales: {day['sales']}")
        print(f"  Agent Action: {day['agent_action']}")
        print(f"  Restock: {day.get('restock', {})}")
        if day.get("new_prices"):
            print(f"  New Prices: {day['new_prices']}")
        if day.get("emails_sent"):
            print(f"  Emails Sent: {day['emails_sent']}")
        print("-" * 40)

Day 1:
  Balance: $98.75
  Inventory: {'Red Bull': 0, 'Snickers': 0}
  Prices: {'Red Bull': 2.5, 'Snickers': 1.25}
  Sales: {'Red Bull': 1, 'Snickers': 1}
  Agent Action: restock Red Bull 20, restock Snickers 40.
  Restock: {'Red Bull': 20, 'Snickers': 40}
----------------------------------------
Day 2:
  Balance: $56.25
  Inventory: {'Red Bull': 18, 'Snickers': 34}
  Prices: {'Red Bull': 2.5, 'Snickers': 1.25}
  Sales: {'Red Bull': 2, 'Snickers': 6}
  Agent Action: restock Red Bull 10, restock Snickers 10.
  Restock: {'Red Bull': 10, 'Snickers': 10}
----------------------------------------
Day 3:
  Balance: $41.25
  Inventory: {'Red Bull': 26, 'Snickers': 40}
  Prices: {'Red Bull': 2.5, 'Snickers': 1.25}
  Sales: {'Red Bull': 2, 'Snickers': 4}
  Agent Action: restock Red Bull 10, restock Snickers 20.
  Restock: {'Red Bull': 10, 'Snickers': 20}
----------------------------------------
